# Generalized Maximum Entropy — reproduction of Figures 2, 3 and 4

This notebook reproduces the numerical results of *Generalized Maximum Entropy:
When and Why You Need It* (Ferro, Pos & Somazzi).

- **Figure 2** — histogram of a single simulated sample with the inferred
  distribution, and the log-likelihood contour around the MLE.
- **Figure 3** — sensitivity of the inferred parameters (q, ψ) to the number of
  realizations *n*, the chain length *N*, and the number of flips *d*.
- **Figure 4** — parameter recovery: inferred q* vs. the true generating q.

The core model functions are identical to those used for Figure 2; Figures 3
and 4 reuse them. For speed, the log-likelihood computes the partition function
once per evaluation (the naive per-observation recomputation is mathematically
identical but far slower, which matters because Figs 3–4 require hundreds of
fits).

In [ ]:
import numpy as np
from scipy.special import factorial
from scipy.optimize import minimize
import matplotlib as mpl
import matplotlib.pyplot as plt

mpl.rcParams.update({
    "figure.dpi": 300, "savefig.dpi": 300,
    "pdf.fonttype": 42, "ps.fonttype": 42,
    "axes.linewidth": 0.8, "axes.labelsize": 11,
    "xtick.labelsize": 10, "ytick.labelsize": 10,
    "legend.frameon": False, "legend.fontsize": 10,
    "axes.spines.right": False, "axes.spines.top": False,
})

## Model functions

In [ ]:
def k(d):
    """Normalization constant kappa(d) in Omega_d(N, M)."""
    d = int(d)
    return d**(d - 2) * factorial(np.floor(d / 2)) * factorial(np.floor((d - 1) / 2))

def omega_th_vec(N, delta):
    """Number of configurations Omega_d(N, M) for all M = 0..N (vectorized)."""
    M = np.arange(0, N + 1, 1.0)
    om = (N**(delta - 1) / k(delta)) * (1 - 4 * ((M - 0.5 * N) / N)**2)**(np.floor((delta - 1) / 2))
    return M, om

def num_vec(q, psi, M):
    """Unnormalized q-exponential weight [1-(1-q) psi M]_+^{1/(1-q)} (vectorized)."""
    if q == 1:
        return np.exp(-psi * M)
    base = 1 - (1 - q) * psi * M
    out = np.zeros_like(M, dtype=float)
    pos = base > 0
    out[pos] = base[pos]**(1 / (1 - q))
    return out

def pmf_over_M(q, psi, N, delta):
    """Normalized probability mass function p(M) = Omega_d(N,M) * weight / Z."""
    M_all, om = omega_th_vec(N, delta)
    w = np.clip(om * num_vec(q, psi, M_all), 0, None)
    s = w.sum()
    return M_all, (w / s if s > 0 else w)

def make_neg_loglik(N, delta):
    """Return an average negative log-likelihood function nll(x, counts) for a
    given (N, delta). `counts` is the histogram of observed M over 0..N.
    Z is computed once per call (fast, identical to the per-sample version)."""
    M_all, om = omega_th_vec(N, delta)
    def nll(x, counts):
        q, psi = x
        w = om * num_vec(q, psi, M_all)
        Z = w.sum()
        if Z <= 0:
            return np.inf
        pmf = w / Z
        mask = pmf > 0
        if np.any(counts[~mask] > 0):     # observed an M with zero model prob
            return np.inf
        return -np.sum(counts[mask] * np.log(pmf[mask])) / counts.sum()
    return nll, M_all, om

def fit_mle(counts, nll, x0=(1.1, 1.4)):
    """Maximum-likelihood (q*, psi*) by Nelder-Mead."""
    res = minimize(nll, list(x0), args=(counts,),
                   bounds=[(1e-12, None), (None, None)],
                   method="nelder-mead", tol=1e-6, options={"maxiter": 400})
    return res.x[0], res.x[1]

def sample_counts(q, psi, N, delta, n, rng):
    """Draw n i.i.d. M-values from p(M; q, psi) and return their histogram."""
    M_all, pmf = pmf_over_M(q, psi, N, delta)
    M_emp = rng.choice(M_all.astype(int), n, p=pmf)
    return np.bincount(M_emp, minlength=N + 1).astype(float)

## Reference configuration

As in the paper: chain length `N = 100`, flips `d = 4`, and a generating
distribution with `q = 1.8`, `psi = 0.9` (the values behind Figure 2). The
reference sample size is `n = 1000`.

In [ ]:
N_ref, d_ref, n_ref = 100, 4, 1000
q_gen, psi_gen = 1.8, 0.9   # generating distribution used for Figs 2 and 3

## Figure 3 — sensitivity to n, N, d

For each swept value we draw `R` independent samples, fit `(q*, psi*)` to each,
and report the median and interquartile range. The red markers give the
fraction of psi estimates above the (arbitrary) threshold psi = 3, which fall
off the plotted range.

> The paper uses `R = 300`. That takes a few minutes. Set `R_FIG3 = 300` to
> reproduce the published figure exactly; a smaller value (e.g. 50) gives a
> quick preview.

In [ ]:
R_FIG3 = 300   # set to 300 for the published figure; lower is faster

n_grid = [50, 100, 250, 500, 1000, 2500, 5000]
N_grid = [25, 50, 100, 200, 400]
d_grid = [2, 3, 4, 5, 6]
PSI_THRESH = 3.0

def sweep(param, values, R, seed=12345):
    med_q, med_psi = [], []
    iqr_q, iqr_psi = [], []
    frac_off, all_q, all_psi = [], [], []
    for v in values:
        N = v if param == "N" else N_ref
        d = v if param == "d" else d_ref
        n = v if param == "n" else n_ref
        nll, _, _ = make_neg_loglik(N, d)
        rng = np.random.default_rng(seed)
        qs, ps = [], []
        for _ in range(R):
            c = sample_counts(q_gen, psi_gen, N, d, n, rng)
            qh, ph = fit_mle(c, nll)
            qs.append(qh); ps.append(ph)
        qs, ps = np.array(qs), np.array(ps)
        med_q.append(np.median(qs)); med_psi.append(np.median(ps))
        iqr_q.append(np.percentile(qs, [25, 75]))
        iqr_psi.append(np.percentile(ps, [25, 75]))
        frac_off.append(np.mean(ps > PSI_THRESH))
        all_q.append(qs); all_psi.append(ps)
    return dict(values=values, med_q=med_q, med_psi=med_psi,
                iqr_q=np.array(iqr_q), iqr_psi=np.array(iqr_psi),
                frac_off=frac_off, all_q=all_q, all_psi=all_psi)

res_n = sweep("n", n_grid, R_FIG3)
res_N = sweep("N", N_grid, R_FIG3)
res_d = sweep("d", d_grid, R_FIG3)

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(9.0, 5.2), sharex="col")
cols = [("n", "number of realizations $n$", res_n),
        ("N", "chain length $N$", res_N),
        ("d", "number of flips $d$", res_d)]

for j, (param, xlabel, res) in enumerate(cols):
    xs = np.array(res["values"], dtype=float)
    xpos = np.arange(len(xs))   # even spacing on a categorical axis

    # ---- top row: inferred q ----
    axq = axes[0, j]
    for i, qs in enumerate(res["all_q"]):
        axq.scatter(np.full_like(qs, xpos[i]), qs, s=5, color="0.7",
                    alpha=0.35, edgecolors="none", zorder=1)
    axq.fill_between(xpos, res["iqr_q"][:, 0], res["iqr_q"][:, 1],
                     color="tab:blue", alpha=0.20, zorder=2)
    axq.plot(xpos, res["med_q"], "-o", color="tab:blue", ms=4, lw=1.2, zorder=3)
    ref_val = {"n": n_ref, "N": N_ref, "d": d_ref}[param]
    if ref_val in res["values"]:
        axq.axvline(res["values"].index(ref_val), ls=":", lw=0.9, color="0.4")
    axq.set_xticks(xpos)
    axq.set_xticklabels([f"{int(v)}" for v in xs])
    if j == 0: axq.set_ylabel("inferred $q$")

    # ---- bottom row: inferred psi ----
    axp = axes[1, j]
    for i, ps in enumerate(res["all_psi"]):
        ps_in = ps[ps <= PSI_THRESH]
        axp.scatter(np.full_like(ps_in, xpos[i]), ps_in, s=5, color="0.7",
                    alpha=0.35, edgecolors="none", zorder=1)
    axp.fill_between(xpos, res["iqr_psi"][:, 0], res["iqr_psi"][:, 1],
                     color="tab:blue", alpha=0.20, zorder=2)
    axp.plot(xpos, res["med_psi"], "-o", color="tab:blue", ms=4, lw=1.2, zorder=3)
    if ref_val in res["values"]:
        axp.axvline(res["values"].index(ref_val), ls=":", lw=0.9, color="0.4")
    # red off-scale fractions
    ymax = PSI_THRESH
    for i, f in enumerate(res["frac_off"]):
        axp.plot(xpos[i], ymax*0.97, marker="^", color="tab:red", ms=6, clip_on=False)
        axp.text(xpos[i], ymax*1.02, f"{100*f:.0f}%", color="tab:red",
                 ha="center", va="bottom", fontsize=7)
    axp.set_ylim(0, PSI_THRESH)
    axp.set_xticks(xpos)
    axp.set_xticklabels([f"{int(v)}" for v in xs])
    axp.set_xlabel(xlabel)
    if j == 0: axp.set_ylabel("inferred $\\psi$")

fig.tight_layout()
fig.savefig("sensitivity_main.pdf")
fig.savefig("sensitivity_main.png", dpi=300)
plt.show()

## Figure 4 — parameter recovery

We draw samples from the model at a range of known `q_true` (holding
`psi_true = 0.05`, `n = 1000`), re-fit, and check that the inferred `q*`
recovers `q_true` across the compact (`q<1`), exponential (`q=1`) and
heavy-tailed (`q>1`) regimes.

> The paper uses `R = 200`. Set `R_FIG4 = 200` to reproduce it exactly.

In [ ]:
R_FIG4 = 200   # set to 200 for the published figure; lower is faster
psi_true = 0.05
q_true_grid = np.round(np.arange(0.6, 1.95, 0.1), 2)

nll4, M_all4, _ = make_neg_loglik(N_ref, d_ref)
med_q, iqr_q, med_p, iqr_p = [], [], [], []
for qt in q_true_grid:
    rng = np.random.default_rng(999)
    qs, ps = [], []
    for _ in range(R_FIG4):
        c = sample_counts(qt, psi_true, N_ref, d_ref, n_ref, rng)
        qh, ph = fit_mle(c, nll4)
        qs.append(qh); ps.append(ph)
    qs, ps = np.array(qs), np.array(ps)
    med_q.append(np.median(qs)); iqr_q.append(np.percentile(qs, [25, 75]))
    med_p.append(np.median(ps)); iqr_p.append(np.percentile(ps, [25, 75]))
iqr_q, iqr_p = np.array(iqr_q), np.array(iqr_p)

In [ ]:
fig, (axq, axp) = plt.subplots(2, 1, figsize=(5.4, 5.4), sharex=True)

# top: inferred q* vs q_true
axq.fill_between(q_true_grid, iqr_q[:, 0], iqr_q[:, 1],
                 color="tab:blue", alpha=0.20)
axq.plot(q_true_grid, med_q, "-o", color="tab:blue", ms=4, lw=1.2,
         label="median inferred $q^*$")
axq.plot(q_true_grid, q_true_grid, "k--", lw=1.0, label="$q^*=q_{\\mathrm{true}}$")
axq.axvline(1.0, ls=":", lw=0.9, color="0.4")
axq.set_ylabel("inferred $q^*$")
axq.legend(loc="upper left")

# bottom: inferred psi*
axp.fill_between(q_true_grid, iqr_p[:, 0], iqr_p[:, 1],
                 color="tab:blue", alpha=0.20)
axp.plot(q_true_grid, med_p, "-o", color="tab:blue", ms=4, lw=1.2,
         label="median inferred $\\psi^*$")
axp.axhline(psi_true, ls="--", color="k", lw=1.0, label="$\\psi_{\\mathrm{true}}=0.05$")
axp.axvline(1.0, ls=":", lw=0.9, color="0.4")
axp.set_ylim(0, 0.10)
axp.set_xlabel("true entropic index $q_{\\mathrm{true}}$")
axp.set_ylabel("inferred $\\psi^*$")
axp.legend(loc="upper left")

fig.tight_layout()
fig.savefig("recovery_qtrue.pdf")
fig.savefig("recovery_qtrue.png", dpi=300)
plt.show()